# 피부질환 10개 클래스 — EfficientNet-B0 학습

Google Drive의 `MyDrive/skin_dataset/processed.zip`을 사용합니다.

ZIP 내부:
```text
processed/
├── original/
│   ├── train/
│   ├── val/
│   └── test/
└── augmented/
    ├── train/
    ├── val/
    └── test/
```

설정:
- 10개 클래스
- Original: 730/class train, 100/class val, 70/class test
- Augmented: 1460/class train, 100/class val, 70/class test
- EfficientNet-B0, ImageNet pretrained
- 입력 224×224
- 정확히 15 Epoch
- Early Stopping 없음
- Validation Accuracy 최고 모델 저장
- 한글 폰트 설정
- Accuracy/Loss, Confusion Matrix, Classification Report 자동 저장
- Original vs Augmented 비교


In [ ]:
# 1. Google Drive 연결
from google.colab import drive
drive.mount("/content/drive")
print("Google Drive 연결 완료")


In [ ]:
# 2. ZIP 확인
from pathlib import Path
import os, shutil, zipfile, json

ZIP_PATH = Path("/content/drive/MyDrive/skin_dataset/processed.zip")

if not ZIP_PATH.exists():
    raise FileNotFoundError(f"processed.zip을 찾을 수 없습니다:\n{ZIP_PATH}")

print("ZIP 확인:", ZIP_PATH)
print(f"크기: {ZIP_PATH.stat().st_size / (1024**3):.2f} GB")


In [ ]:
# 3. ZIP을 Colab 로컬로 복사 후 압축 해제
LOCAL_ZIP = Path("/content/processed.zip")
EXTRACT_ROOT = Path("/content/skin_dataset")

if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)

print("Google Drive → Colab 로컬 ZIP 복사 중...")
shutil.copy2(ZIP_PATH, LOCAL_ZIP)

print("압축 해제 중...")
with zipfile.ZipFile(LOCAL_ZIP, "r") as z:
    z.extractall(EXTRACT_ROOT)

print("압축 해제 완료")


In [ ]:
# 4. 데이터 경로 설정
PROCESSED_ROOT = EXTRACT_ROOT / "processed"
ORIGINAL_ROOT = PROCESSED_ROOT / "original"
AUGMENTED_ROOT = PROCESSED_ROOT / "augmented"

for p in [
    ORIGINAL_ROOT / "train",
    ORIGINAL_ROOT / "val",
    ORIGINAL_ROOT / "test",
    AUGMENTED_ROOT / "train",
    AUGMENTED_ROOT / "val",
    AUGMENTED_ROOT / "test",
]:
    print("✅" if p.exists() else "❌", p)

if not ORIGINAL_ROOT.exists() or not AUGMENTED_ROOT.exists():
    raise FileNotFoundError("processed/original 또는 processed/augmented 구조를 찾지 못했습니다.")


In [ ]:
# 5. 라이브러리 + 한글 폰트 + GPU
!apt-get update -qq
!apt-get install -y fonts-nanum -qq

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import tensorflow as tf

from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.applications import EfficientNetB0

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if Path(font_path).exists():
    plt.rcParams["font.family"] = fm.FontProperties(fname=font_path).get_name()
    plt.rcParams["axes.unicode_minus"] = False
    print("한글 폰트 설정 완료")

print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus if gpus else "⚠️ 없음")


In [ ]:
# 6. 학습 설정
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 15
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

print(f"입력: {IMG_SIZE}")
print(f"Batch: {BATCH_SIZE}")
print(f"Epoch: {EPOCHS}")
print("Early Stopping: 사용 안 함")


In [ ]:
# 7. 클래스 확인
class_names = sorted([
    p.name for p in (ORIGINAL_ROOT / "train").iterdir()
    if p.is_dir()
])

print("클래스 수:", len(class_names))
for i, name in enumerate(class_names):
    print(f"{i}: {name}")

if len(class_names) != 10:
    print("⚠️ 예상 클래스 수는 10개입니다.")


In [ ]:
# 8. 데이터 개수 확인
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def count_images(folder):
    folder = Path(folder)
    if not folder.exists():
        print("❌ 없음:", folder)
        return 0
    total = 0
    for d in sorted(folder.iterdir()):
        if d.is_dir():
            n = sum(1 for p in d.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
            print(f"{d.name}: {n}장")
            total += n
    print("전체:", total)
    return total

for name, root in [("ORIGINAL", ORIGINAL_ROOT), ("AUGMENTED", AUGMENTED_ROOT)]:
    print("\n" + "="*60)
    print(name)
    for split in ["train", "val", "test"]:
        print(f"\n[{split}]")
        count_images(root / split)


In [ ]:
# 9. TensorFlow Dataset
def create_datasets(root):
    root = Path(root)

    train_ds = tf.keras.utils.image_dataset_from_directory(
        root / "train",
        labels="inferred",
        label_mode="int",
        class_names=class_names,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=True,
        seed=SEED
    )

    val_ds = tf.keras.utils.image_dataset_from_directory(
        root / "val",
        labels="inferred",
        label_mode="int",
        class_names=class_names,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    test_ds = tf.keras.utils.image_dataset_from_directory(
        root / "test",
        labels="inferred",
        label_mode="int",
        class_names=class_names,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    return (
        train_ds.prefetch(AUTOTUNE),
        val_ds.prefetch(AUTOTUNE),
        test_ds.prefetch(AUTOTUNE)
    )


In [ ]:
# 10. EfficientNet-B0
def build_model():
    base = EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=(224, 224, 3)
    )
    base.trainable = False

    inputs = layers.Input(shape=(224, 224, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.30)(x)
    outputs = layers.Dense(len(class_names), activation="softmax")(x)

    model = models.Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


In [ ]:
# 11. 학습 + 평가
def train_and_evaluate(dataset_name, root):
    print("\n" + "="*70)
    print(f"{dataset_name} 학습 시작")
    print("="*70)

    train_ds, val_ds, test_ds = create_datasets(root)
    model = build_model()

    result_dir = Path(f"/content/results_{dataset_name.lower()}")
    result_dir.mkdir(parents=True, exist_ok=True)
    model_path = result_dir / "best_model.keras"

    checkpoint = callbacks.ModelCheckpoint(
        filepath=str(model_path),
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    )

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=[checkpoint]
    )

    # Validation Accuracy가 가장 높았던 모델 사용
    model = tf.keras.models.load_model(model_path)

    test_loss, test_accuracy = model.evaluate(test_ds, verbose=1)

    print(f"\n{dataset_name} Test Accuracy: {test_accuracy*100:.2f}%")

    # Accuracy
    plt.figure(figsize=(8, 6))
    plt.plot(history.history["accuracy"], label="Train Accuracy")
    plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"{dataset_name} Accuracy")
    plt.legend()
    plt.grid()
    plt.tight_layout()
    plt.savefig(result_dir / "accuracy.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    # Loss
    plt.figure(figsize=(8, 6))
    plt.plot(history.history["loss"], label="Train Loss")
    plt.plot(history.history["val_loss"], label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{dataset_name} Loss")
    plt.legend()
    plt.grid()
    plt.tight_layout()
    plt.savefig(result_dir / "loss.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    # Test prediction
    y_true, y_pred = [], []
    for images, labels in test_ds:
        pred = model.predict(images, verbose=0)
        y_true.extend(labels.numpy())
        y_pred.extend(np.argmax(pred, axis=1))

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(11, 9))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names
    )
    plt.xlabel("예측 클래스")
    plt.ylabel("실제 클래스")
    plt.title(f"{dataset_name} Confusion Matrix")
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(result_dir / "confusion_matrix.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    # Classification report
    report = classification_report(
        y_true, y_pred,
        target_names=class_names,
        digits=4
    )
    print("\n" + "="*70)
    print(f"{dataset_name} Classification Report")
    print("="*70)
    print(report)

    with open(result_dir / "classification_report.txt", "w", encoding="utf-8") as f:
        f.write(report)

    best_val_accuracy = max(history.history["val_accuracy"])
    best_epoch = int(np.argmax(history.history["val_accuracy"]) + 1)

    result = {
        "dataset": dataset_name,
        "model": "EfficientNetB0",
        "image_size": [224, 224],
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "early_stopping": False,
        "best_val_accuracy": float(best_val_accuracy),
        "best_epoch": best_epoch,
        "test_loss": float(test_loss),
        "test_accuracy": float(test_accuracy),
        "classes": class_names
    }

    with open(result_dir / "results.json", "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=4)

    print(f"최고 Val Accuracy: {best_val_accuracy*100:.2f}% (Epoch {best_epoch})")
    print("결과 저장:", result_dir)

    return result


# 12. Original 모델 학습

정확히 15 Epoch 학습합니다.
Early Stopping은 사용하지 않습니다.


In [ ]:
original_result = train_and_evaluate(
    "Original",
    ORIGINAL_ROOT
)


# 13. Augmented 모델 학습

정확히 15 Epoch 학습합니다.
Early Stopping은 사용하지 않습니다.


In [ ]:
augmented_result = train_and_evaluate(
    "Augmented",
    AUGMENTED_ROOT
)


In [ ]:
# 14. Original vs Augmented 비교
original_acc = original_result["test_accuracy"]
augmented_acc = augmented_result["test_accuracy"]
difference = augmented_acc - original_acc

print("\n" + "="*70)
print("Original vs Augmented")
print("="*70)
print(f"Original  Test Accuracy : {original_acc*100:.2f}%")
print(f"Augmented Test Accuracy : {augmented_acc*100:.2f}%")
print(f"차이                    : {difference*100:+.2f}%p")
print()
print(f"Original 최고 Val Accuracy : {original_result['best_val_accuracy']*100:.2f}% "
      f"(Epoch {original_result['best_epoch']})")
print(f"Augmented 최고 Val Accuracy: {augmented_result['best_val_accuracy']*100:.2f}% "
      f"(Epoch {augmented_result['best_epoch']})")


In [ ]:
# 15. 결과 ZIP 생성
RESULT_ROOT = Path("/content/model_results")

if RESULT_ROOT.exists():
    shutil.rmtree(RESULT_ROOT)

RESULT_ROOT.mkdir(parents=True)

shutil.copytree(
    "/content/results_original",
    RESULT_ROOT / "results_original"
)

shutil.copytree(
    "/content/results_augmented",
    RESULT_ROOT / "results_augmented"
)

zip_result = shutil.make_archive(
    "/content/skin_model_results",
    "zip",
    RESULT_ROOT
)

print("✅ 결과 ZIP:", zip_result)


In [ ]:
# 16. 결과 ZIP 다운로드
from google.colab import files

files.download("/content/skin_model_results.zip")
